In [2]:
## CPU Usage Config

import os
import torch

CPU_THREADS = 4

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

torch.set_num_threads(CPU_THREADS)

print("Using CPU threads:", CPU_THREADS)

Using CPU threads: 4


In [5]:
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

# ==========================
# USER CONFIG
# ==========================
DATA_YAML = Path("/workspace/flower-classifier/config/data.yaml")
DATASET_ROOT = Path("/workspace/data/data_set/tomato-flower")
PROJECT_DIR = Path("/workspace/flower-classifier/runs")
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"   # change later if needed

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    print(f"{label}:")
    print(f"  path   = {path}")
    print(f"  exists = {path.exists()}")
    print(f"  is_dir = {path.is_dir()}")
    print(f"  is_file= {path.is_file()}")
    print()

# ==========================
# SYSTEM INFO
# ==========================
print("=== SYSTEM INFO ===")
print("Python executable:", sys.executable)
print("Python version   :", sys.version.splitlines()[0])
print("Platform         :", platform.platform())
print("Machine          :", platform.machine())
print("Processor        :", platform.processor())
print("CPU count        :", multiprocessing.cpu_count())
print("Working dir      :", Path.cwd())
print()

print("=== TORCH INFO ===")
print("torch version    :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("MPS available    :", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("torch num threads:", torch.get_num_threads())
print()

# ==========================
# ENV VARS
# ==========================
print("=== IMPORTANT ENV VARS ===")
for key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "CUDA_VISIBLE_DEVICES",
    "ULTRALYTICS_CONFIG_DIR",
]:
    print(f"{key} =", os.environ.get(key))
print()

# ==========================
# PATH CHECKS
# ==========================
print("=== PATH CHECKS ===")
print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)
    print(f"{split.upper()} COUNTS:")
    print("  image files =", count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}))
    print("  label files =", count_files(lbl_dir, {".txt"}))
    print()

# ==========================
# YAML CHECK
# ==========================
print("=== YAML CONTENTS ===")
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print(json.dumps(data_cfg, indent=2))

print()
print("=== YAML PATH VALIDATION ===")
for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    print(f"{key}: {p} | exists={p.exists()} | is_dir={p.is_dir()}")

=== SYSTEM INFO ===
Python executable: /usr/local/bin/python3.11
Python version   : 3.11.15 (main, Mar 16 2026, 23:07:56) [GCC 14.2.0]
Platform         : Linux-6.17.0-19-generic-x86_64-with-glibc2.41
Machine          : x86_64
Processor        : 
CPU count        : 8
Working dir      : /workspace/flower-classifier/notebooks

=== TORCH INFO ===
torch version    : 2.11.0+cu130
CUDA available   : False
MPS available    : False
torch num threads: 4

=== IMPORTANT ENV VARS ===
OMP_NUM_THREADS = 4
MKL_NUM_THREADS = 4
NUMEXPR_NUM_THREADS = 4
OPENBLAS_NUM_THREADS = 4
VECLIB_MAXIMUM_THREADS = None
CUDA_VISIBLE_DEVICES = None
ULTRALYTICS_CONFIG_DIR = None

=== PATH CHECKS ===
DATA_YAML:
  path   = /workspace/flower-classifier/config/data.yaml
  exists = True
  is_dir = False
  is_file= True

DATASET_ROOT:
  path   = /workspace/data/data_set/tomato-flower
  exists = True
  is_dir = True
  is_file= False

PROJECT_DIR:
  path   = /workspace/flower-classifier/runs
  exists = True
  is_dir = True
  is

In [6]:
## Verify YOLO26 is accessible and usable

from ultralytics import YOLO

try:
    YOLO("yolo26n.yaml")
    print("DONE! Model config recognized")
except Exception as e:
    print("ERROR! Model config NOT recognized")
    print(e)

DONE! Model config recognized
